# Setup

In [2]:
from pystac_client import Client
from collections import defaultdict
from io import BytesIO
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
from PIL import Image
import requests

import numpy as np

gdal_data_paths = [
    Path(sys.prefix) / "Library" / "share" / "gdal",
    Path(sys.prefix) / "share" / "gdal"]

gdal_data_path = next(
    (path for path in gdal_data_paths
     if (path / "gdalvrt.xsd").exists()),
    None)

if gdal_data_path is not None:
    os.environ["GDAL_DATA"] = str(gdal_data_path)

print("GDAL data:", gdal_data_path)

import rasterio
from rasterio.merge import merge
from rasterio.warp import transform_bounds

GDAL data: c:\Users\harry\anaconda3\envs\KGG375_AT3\Library\share\gdal


In [3]:
from pathlib import Path
import rasterio

tile_dir = Path("data/processed/sar_2019_tiles")

for tile_path in sorted(tile_dir.glob("*.tif")):
    print("Checking:", tile_path.name, flush = True)

    with rasterio.open(tile_path) as source:
        source.read(1, window = ((0, 16), (0, 16)))
        print(source.width, source.height, source.count, flush = True)

Checking: S1A_2019-04-07_RTC_10m_r0_c0.tif
2400 2400 3
Checking: S1A_2019-04-07_RTC_10m_r0_c2400.tif
342 2400 3
Checking: S1A_2019-04-07_RTC_10m_r2400_c0.tif
2400 989 3
Checking: S1A_2019-04-07_RTC_10m_r2400_c2400.tif
342 989 3


C:\Users\harry\AppData\Local\Temp\ipykernel_29968\1080206710.py:10: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source.read(1, window = ((0, 16), (0, 16)))


## Source Imagery for Outlines

In [ ]:
catalog = Client.open("https://stac.dataspace.copernicus.eu/v1/")

search = catalog.search(
    collections = ["sentinel-2-l1c"],
    bbox = [73.45, -53.25, 73.85, -52.95],
    datetime = "2019-04-01/2019-04-14",
    method = "GET",
    limit = 5,
    max_items = 5)

items = list(search.items())

print(f"Returned {len(items)} scenes (maximum 5)")
for item in items:
    print(item.datetime.date(), item.properties.get("eo:cloud_cover"), item.id)

In [ ]:
search = catalog.search(
    collections = ["sentinel-2-l1c"],
    bbox = [73.45, -53.25, 73.85, -52.95],
    datetime = "2019-04-01/2019-04-14",
    method = "GET",
    limit = 10,
    max_items = 80)

items = list(search.items())
by_acquisition = defaultdict(list)

for item in items:
    by_acquisition[item.datetime.strftime("%Y-%m-%d %H:%M")].append(item)

print(f"{len(items)} tiles across {len(by_acquisition)} acquisition times")

for when, tiles in sorted(by_acquisition.items()):
    details = [
        (item.id.split("_")[5], item.properties.get("eo:cloud_cover"))
        for item in tiles]
    print(when, details)

if len(items) == 80:
    print("Result cap reached: search a shorter date window.")

In [ ]:
april6 = [
    item for item in items
    if item.datetime.date().isoformat() == "2019-04-06"]

for item in april6:
    tile = item.id.split("_")[5]
    print(tile, [round(value, 3) for value in item.bbox])

print("Preview/asset keys:", list(april6[0].assets))

In [ ]:
tile_items = sorted(
    [item for item in items if "_T43FDB_" in item.id],
    key=lambda item: item.datetime)

with plt.style.context("dark_background"):
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    for ax, item in zip(axes.flat, tile_items):
        response = requests.get(item.assets["thumbnail"].href, timeout=30)
        response.raise_for_status()

        cloud = item.properties["eo:cloud_cover"]
        ax.imshow(Image.open(BytesIO(response.content)))
        ax.set_title(f"{item.datetime:%d %b} | {cloud:.0f}% tile cloud")
        ax.axis("off")

    fig.suptitle("Sentinel-2 previews: T43FDB, April 2019")
    fig.tight_layout()
    plt.show()

## SAR Data

In [5]:
# Search Sentinel-1 GRD acquisitions

heard_bbox = [73.2, -53.3, 73.9, -52.85]

s1_search = catalog.search(
    collections = ["sentinel-1-grd"],
    bbox = heard_bbox,
    datetime = "2019-03-01/2019-04-30",
    method = "GET",
    limit = 10,
    max_items = 100)

s1_items = sorted(
    s1_search.items(),
    key = lambda item: item.datetime)

print(f"Returned {len(s1_items)} Sentinel-1 products\n")

for item in s1_items:
    print(
        item.datetime.strftime("%Y-%m-%d %H:%M"),
        item.properties.get("sar:instrument_mode"),
        item.properties.get("sat:orbit_state"),
        item.properties.get("s1:polarization"),
        item.id)

NameError: name 'catalog' is not defined

In [ ]:
# Inspect acquisitions around the Pléiades date

candidate_dates = {
    "2019-04-06",
    "2019-04-07",
    "2019-04-08"}

candidate_items = [
    item for item in s1_items
    if item.datetime.date().isoformat() in candidate_dates]

for item in candidate_items:
    polarisation = item.properties.get(
        "sar:polarizations",
        item.id.split("_")[4])

    print(item.datetime.strftime("%Y-%m-%d %H:%M"))
    print("Mode:", item.properties.get("sar:instrument_mode"))
    print("Orbit:", item.properties.get("sat:orbit_state"))
    print("Polarisation:", polarisation)
    print("Bounding box:", [round(value, 3) for value in item.bbox])
    print("Assets:", list(item.assets))
    print()

In [ ]:
# Compare candidate Sentinel-1 previews
unique_candidates = {}

for item in candidate_items:
    acquisition = item.datetime.isoformat()
    unique_candidates.setdefault(acquisition, item)

candidate_items = sorted(
    unique_candidates.values(),
    key = lambda item: item.datetime)

with plt.style.context("dark_background"):
    fig, axes = plt.subplots(1, 3, figsize = (15, 6))

    for ax, item in zip(axes, candidate_items):
        response = requests.get(
            item.assets["thumbnail"].href,
            timeout = 30)
        response.raise_for_status()

        mode = item.properties.get("sar:instrument_mode")
        orbit = item.properties.get("sat:orbit_state")

        ax.imshow(Image.open(BytesIO(response.content)))
        ax.set_title(
            f"{item.datetime:%d %b %Y}\n"
            f"{mode} | {orbit.title()}")
        ax.axis("off")

    fig.suptitle("Sentinel-1 candidate acquisitions")
    fig.tight_layout()
    plt.show()

april7 = next(
    item for item in candidate_items
    if item.datetime.date().isoformat() == "2019-04-07")

print("7 April HH asset:")
print(april7.assets["hh"].href)
print("\nMedia type:")
print(april7.assets["hh"].media_type)

In [ ]:
# Find matching Sentinel-1 scenes from 2026

target_orbit = april7.properties.get("sat:relative_orbit")

print("2019 relative orbit:", target_orbit)

s1_2026_search = catalog.search(
    collections = ["sentinel-1-grd"],
    bbox = heard_bbox,
    datetime = "2026-03-01/2026-04-30",
    method = "GET",
    limit = 10,
    max_items = 100)

s1_2026_items = sorted(
    s1_2026_search.items(),
    key = lambda item: item.datetime)

matching_2026 = [
    item for item in s1_2026_items
    if item.properties.get("sar:instrument_mode") == "SM"
    and item.properties.get("sat:orbit_state") == "descending"
    and "HH" in item.properties.get("sar:polarizations", [])
    and (
        target_orbit is None
        or item.properties.get("sat:relative_orbit") == target_orbit)]

print(f"Returned {len(s1_2026_items)} total 2026 products")
print(f"Found {len(matching_2026)} matching Stripmap products\n")

for item in matching_2026:
    print(
        item.datetime.strftime("%Y-%m-%d %H:%M"),
        item.properties.get("platform"),
        item.properties.get("sat:relative_orbit"),
        [round(value, 3) for value in item.bbox],
        item.id)

In [ ]:
# Authenticate with the Sentinel Hub Process API

from getpass import getpass

import requests

client_id = getpass("Client ID: ").strip()
client_secret = getpass("Client secret: ").strip()

if not client_id or not client_secret:
    raise ValueError("Both credentials must be entered")

token_url = (
    "https://identity.dataspace.copernicus.eu/auth/realms/"
    "CDSE/protocol/openid-connect/token")

token_response = requests.post(
    token_url,
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"},
    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret},
    timeout = 30)

if not token_response.ok:
    try:
        error_detail = token_response.json()
    except ValueError:
        error_detail = token_response.text[:300]

    raise RuntimeError(
        f"Authentication failed ({token_response.status_code}): "
        f"{error_detail}")

access_token = token_response.json()["access_token"]

print("Authentication successful")

In [6]:
tile_dir = Path("data/processed/sar_2019_tiles")
tile_paths = sorted(tile_dir.glob("S1A_2019-04-07_RTC_10m_*.tif"))

if len(tile_paths) != 4:
    raise RuntimeError(f"Expected 4 tiles, found {len(tile_paths)}")

output_path = Path("data/processed/S1A_HH_2019-04-07_RTC_10m.tif")

print("Mosaicking four tiles directly to disk...", flush = True)

merge(
    tile_paths,
    dst_path = output_path,
    mem_limit = 64,
    dst_kwds = {
        "compress": "deflate",
        "predictor": 3,
        "tiled": True,
        "BIGTIFF": "IF_SAFER"})

with rasterio.open(output_path) as mosaic:
    shape = (mosaic.count, mosaic.height, mosaic.width)
    print("Output:", output_path)
    print("Shape:", shape)
    print("CRS:", mosaic.crs)
    print("Resolution:", mosaic.res)

    if shape != (3, 3389, 2742):
        raise RuntimeError(f"Unexpected mosaic shape: {shape}")

Mosaicking four tiles directly to disk...


RasterBlockError: The height and width of TIFF dataset blocks must be multiples of 16

In [ ]:
# Process the final 2019 Sentinel-1 raster at 10 m
aoi_wgs84 = [73.45, -53.25, 73.85, -52.95]
pixel_size = 10
maximum_tile_pixels = 2400

aoi_utm = transform_bounds(
    "EPSG:4326",
    "EPSG:32743",
    *aoi_wgs84,
    densify_pts = 21)

min_x, min_y, max_x, max_y = aoi_utm

# Align the complete AOI to the 10 m grid
min_x = np.floor(min_x / pixel_size) * pixel_size
min_y = np.floor(min_y / pixel_size) * pixel_size
max_x = np.ceil(max_x / pixel_size) * pixel_size
max_y = np.ceil(max_y / pixel_size) * pixel_size

full_width = int((max_x - min_x) / pixel_size)
full_height = int((max_y - min_y) / pixel_size)

print("Complete output:", full_width, "×", full_height, "pixels")

evalscript = """
//VERSION=3

function setup() {
    return {
        input: ["HH", "shadowMask", "dataMask"],
        output: {
            id: "default",
            bands: 3,
            sampleType: "FLOAT32"
        }
    }
}

function evaluatePixel(sample) {
    let hhDb = sample.HH > 0
        ? 10 * Math.log(sample.HH) / Math.LN10
        : -99;

    return [hhDb, sample.shadowMask, sample.dataMask];
}
"""

output_dir = Path("data/processed/sar_2019_tiles")
output_dir.mkdir(parents = True, exist_ok = True)

tile_paths = []

for row_start in range(0, full_height, maximum_tile_pixels):
    for column_start in range(0, full_width, maximum_tile_pixels):
        tile_height = min(
            maximum_tile_pixels,
            full_height - row_start)
        tile_width = min(
            maximum_tile_pixels,
            full_width - column_start)

        tile_min_x = min_x + column_start * pixel_size
        tile_min_y = min_y + row_start * pixel_size
        tile_max_x = tile_min_x + tile_width * pixel_size
        tile_max_y = tile_min_y + tile_height * pixel_size

        tile_name = (
            f"S1A_2019-04-07_RTC_10m_"
            f"r{row_start}_c{column_start}.tif")

        tile_path = output_dir / tile_name
        tile_paths.append(tile_path)

        if tile_path.exists():
            print("Already exists:", tile_name)
            continue

        request_body = {
            "input": {
                "bounds": {
                    "bbox": [
                        tile_min_x,
                        tile_min_y,
                        tile_max_x,
                        tile_max_y],
                    "properties": {
                        "crs": (
                            "http://www.opengis.net/def/crs/"
                            "EPSG/0/32743")}},
                "data": [{
                    "type": "sentinel-1-grd",
                    "dataFilter": {
                        "timeRange": {
                            "from": "2019-04-07T00:00:00Z",
                            "to": "2019-04-07T00:02:00Z"},
                        "acquisitionMode": "SM",
                        "polarization": "SH",
                        "orbitDirection": "DESCENDING",
                        "resolution": "HIGH"},
                    "processing": {
                        "backCoeff": "GAMMA0_TERRAIN",
                        "orthorectify": True,
                        "demInstance": "COPERNICUS_30",
                        "radiometricTerrainOversampling": 3,
                        "speckleFilter": {
                            "type": "LEE",
                            "windowSizeX": 5,
                            "windowSizeY": 5}}}]},
            "output": {
                "width": tile_width,
                "height": tile_height,
                "responses": [{
                    "identifier": "default",
                    "format": {
                        "type": "image/tiff"}}]},
            "evalscript": evalscript}

        response = requests.post(
            "https://sh.dataspace.copernicus.eu/process/v1",
            headers = {
                "Authorization": f"Bearer {access_token}",
                "Content-Type": "application/json",
                "Accept": "image/tiff"},
            json = request_body,
            timeout = 300)

        if not response.ok:
            raise RuntimeError(
                f"Tile failed ({response.status_code}): "
                f"{response.text[:1000]}")

        tile_path.write_bytes(response.content)
        print("Downloaded:", tile_name)

# Validate the tile headers
for tile_path in tile_paths:
    size_kb = tile_path.stat().st_size / 1024

    if size_kb == 0:
        raise RuntimeError(f"Empty tile: {tile_path.name}")

    with rasterio.open(tile_path) as source:
        print(
            f"Validated: {tile_path.name} | "
            f"{source.width} × {source.height} | "
            f"{size_kb:,.1f} KB")

# Mosaic directly to disk to limit memory use
output_path = Path(
    "data/processed/S1A_HH_2019-04-07_RTC_10m.tif")

output_options = {
    "compress": "deflate",
    "predictor": 3,
    "tiled": True,
    "BIGTIFF": "IF_SAFER"}

merge(
    tile_paths,
    dst_path = output_path,
    mem_limit = 64,
    dst_kwds = output_options)

with rasterio.open(output_path, "r+") as mosaic:
    mosaic.set_band_description(1, "HH backscatter (dB)")
    mosaic.set_band_description(2, "Radar shadow mask")
    mosaic.set_band_description(3, "Data mask")

    print("\nFinal mosaic:", output_path)
    print("Shape:", (mosaic.count, mosaic.height, mosaic.width))
    print("CRS:", mosaic.crs)
    print("Resolution:", mosaic.res)

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

tile_dir = Path("data/processed/sar_2019_tiles")
tile_paths = sorted(tile_dir.glob("S1A_2019-04-07_RTC_10m_*.tif"))

expected_names = {
    "S1A_2019-04-07_RTC_10m_r0_c0.tif",
    "S1A_2019-04-07_RTC_10m_r0_c2400.tif",
    "S1A_2019-04-07_RTC_10m_r2400_c0.tif",
    "S1A_2019-04-07_RTC_10m_r2400_c2400.tif"}

downloaded_names = {path.name for path in tile_paths}
missing_names = sorted(expected_names - downloaded_names)

print(f"Found {len(tile_paths)} of 4 expected tiles")

for tile_path in tile_paths:
    size_mb = tile_path.stat().st_size / 1024 ** 2
    print(f"{tile_path.name}: {size_mb:.1f} MB")

print("Missing:", missing_names or "none")
print("Python:", sys.executable)
print("GDAL_DATA:", os.environ.get("GDAL_DATA"))

# Test rasterio in a child process so it cannot kill the notebook
if tile_paths:
    probe_code = """
import sys
import rasterio
from rasterio.windows import Window

print(f"Rasterio: {rasterio.__version__}", flush = True)
print(f"GDAL: {rasterio.__gdal_version__}", flush = True)

with rasterio.open(sys.argv[1]) as source:
    source.read(1, window = Window(0, 0, 16, 16))
    print(
        f"Opened successfully: {source.width} × {source.height}",
        flush = True)
"""

    result = subprocess.run(
        [sys.executable, "-c", probe_code, str(tile_paths[0].resolve())],
        capture_output = True,
        text = True)

    print("\nProbe return code:", result.returncode)
    print("Probe output:")
    print(result.stdout or "(none)")
    print("Probe errors:")
    print(result.stderr or "(none)")